# `davao_sovi_svd_montecarlo_validation.ipynb`: SVD Subspace Optimality & Monte Carlo Algorithmic Stability Validation

**Project:** Spatiotemporal Data Analytics and Dimensionality Reduction for Social Vulnerability Assessment in Region XI (2020–2024)  
**Repository:** [`conname/davao-sovi`](https://github.com)  
**Authors:** Cleriza O. Dayot, James Marc M. Barsomo, & Ronalyn L. Mariano (Ateneo de Davao University)  

---

## Notebook Overview

This standalone validation notebook executes **Diagnostics Steps 5 and 6** of the `davao-sovi` robustness protocol. Rather than re-running raw data ingestion, temporal interpolation, or GIS spatial joins, it reads directly from the verified `.csv` artifacts generated by the primary pipeline (`notebooks/davao_sovi_pipeline.ipynb`) to evaluate matrix truncation optimality and numerical stability under small spatial sample constraints (`N = 6` spatial units; `N_T = 30` province-year observations).

---

## Execution Prerequisites & Input Dependencies

Run `notebooks/davao_sovi_pipeline.ipynb` prior to executing this notebook so the following required input files exist in the workspace:

* `data/processed/pooled_z_scores_2020_2024.csv` — Standardized pooled indicator matrix (`30 rows × 15 indicators`).
* `data/processed/Region_XI_SoVI_Panel_Results_2020_2024.csv` — Baseline `SoVI`, rescaled `SoVI_Norm` (`[0, 1]`), and relative `Tier` classifications.
* `outputs/diagnostics/table_4_7_varimax_rotated_loadings.csv` — Sign-aligned Varimax-rotated component loadings (`RC1`–`RC4`).

---

## Analytical Workflow & Diagnostic Modules

### Cell 1: Environment Initialization & CSV Artifact Ingestion
* Imports `numpy`, `pandas`, and `scipy.stats`.
* Loads the pooled $Z$-score matrix ($\mathbf{Z} \in \mathbb{R}^{30 \times 15}$), baseline composite scores, and sign-aligned Varimax reference loadings from `data/processed/` and `outputs/diagnostics/`.

### Cell 2: Diagnostic Step 5 — Eckart–Young–Mirsky SVD Subspace Optimality Bounds
* Computes the Singular Value Decomposition ($\mathbf{Z} = \mathbf{U}\mathbf{\Sigma}\mathbf{V}^\top$) across ranks $k \in \{1, \dots, 15\}$.
* Evaluates cumulative variance explained, Frobenius residual error $\|\mathbf{Z} - \mathbf{Z}_k\|_F$, and trailing eigenvalue decay ($\lambda_k \le 0.3765$ for $k \ge 5$) to confirm that truncating at $m = 4$ components minimizes arbitrary reconstruction noise.
* Executes paired **Wilcoxon signed-rank tests** comparing observation-level reconstruction residuals between under-specified ($k = 3$), selected ($k = 4$), and expanded ($k = 5$) subspaces.

### Cell 3: Diagnostic Step 6 — Monte Carlo Algorithmic Stability Simulation (`B = 10,000`)
* Injects controlled Gaussian perturbations ($\sigma_{\text{noise}} = 0.05\text{ SD}$) into the standardized panel across **$B = 10,000$ independent iterations** to simulate survey sampling and measurement noise.
* Tracks correlation matrix condition numbers ($\kappa(\mathbf{R}) < 30.0$), determinant stability ($|\mathbf{R}|$), Spearman rank correlation ($r_s \ge 0.80$), and Class Mobility Index ($M_{\text{class}} \le 0.15$).
* Compiles observation-level **95% simulation confidence intervals** for rescaled `SoVI'` scores across all 30 province-year units.

---

## Exported Diagnostic CSV Outputs (`outputs/diagnostics/`)

| Output File | Diagnostic Scope | Key Metrics Logged |
| :--- | :--- | :--- |
| `table_4_15_eckart_young_mirsky_svd_bounds.csv` | Rank $k=1 \dots 15$ SVD bounds | Singular values ($\sigma_k$), eigenvalues ($\lambda_k$), cumulative variance (%), Frobenius residual error |
| `table_4_15b_svd_rank_wilcoxon_tests.csv` | Subspace rank comparison | Mean observation-level reconstruction error, Wilcoxon $W$ statistic, and two-tailed $p$-values |
| `table_4_16_monte_carlo_stability_summary.csv` | Global simulation diagnostics | Mean/95% CI of $\kappa(\mathbf{R})$, $|\mathbf{R}|$, Spearman $r_s$, tier match (%), and $M_{\text{class}}$ |
| `table_4_16b_monte_carlo_observation_ci.csv` | Province-year score bands ($N=30$) | Baseline `SoVI_Norm`, simulated mean, SD, and 2.5th–97.5th percentile confidence bounds |


In [4]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP & LOAD EXPORTED CSV FILES
# ==============================================================================
import os
import numpy as np
import pandas as pd
from scipy import stats

# Input & Output Directory (all inputs loaded directly from outputs/diagnostics)
diag_dir = os.path.join("outputs", "diagnostics")
os.makedirs(diag_dir, exist_ok=True)

# 1. Load Pooled Standardized Z-Score Matrix (N = 30 x 15)
z_path = os.path.join(diag_dir, "pooled_z_scores_2020_2024.csv")
df_z = pd.read_csv(z_path)

id_cols = ["Province", "Year"]
cols = [c for c in df_z.columns if c not in id_cols]
Z_mat = df_z[cols].values
N, p = Z_mat.shape

# 2. Load Baseline Panel Results & Sign-Aligned Varimax Loadings
panel_path = os.path.join(diag_dir, "Region_XI_SoVI_Panel_Results_2020_2024.csv")
df_panel = pd.read_csv(panel_path)

loadings_path = os.path.join(diag_dir, "table_4_7_varimax_rotated_loadings.csv")
df_loadings = pd.read_csv(loadings_path, index_col="Indicator")
rc_cols = [c for c in df_loadings.columns if c.startswith("RC")]
load_rot_ref = df_loadings.loc[cols, rc_cols].values
m = len(rc_cols)

print(f"[LOADED] Pooled Z-score matrix: {N} observations x {p} indicators")
print(f"[LOADED] Baseline panel results ({len(df_panel)} rows) & {m}-component rotated loadings")


FileNotFoundError: [Errno 2] No such file or directory: 'outputs\\diagnostics\\pooled_z_scores_2020_2024.csv'

In [ ]:
# ==============================================================================
# CELL 2: STEP 5 — ECKART-YOUNG-MIRSKY SVD SUBSPACE OPTIMALITY BOUNDS
# ==============================================================================
print("=== STEP 5: ECKART-YOUNG-MIRSKY MATRIX APPROXIMATION BOUNDS ===")

# Singular Value Decomposition of standardized matrix Z (N x p)
# Note: Eigenvalues of correlation matrix R satisfy lambda_k = (sigma_k^2) / (N - 1)
U, s, Vt = np.linalg.svd(Z_mat, full_matrices=False)
eigvals_svd = (s ** 2) / (N - 1)
total_frobenius_sq = np.sum(s ** 2)

svd_rows = []
for k in range(1, p + 1):
    # Optimal rank-k reconstruction via truncated SVD
    Z_hat_k = (U[:, :k] * s[:k]) @ Vt[:k, :]
    frob_error_k = np.linalg.norm(Z_mat - Z_hat_k, ord="fro")
    rel_frob_error_pct = (frob_error_k / np.sqrt(total_frobenius_sq)) * 100.0
    var_captured_pct = (np.sum(eigvals_svd[:k]) / p) * 100.0
    tail_max_eig = eigvals_svd[k] if k < p else 0.0

    svd_rows.append({
        "Rank_k": k,
        "Singular_Value_sigma": s[k - 1],
        "Eigenvalue_lambda": eigvals_svd[k - 1],
        "Cumulative_Variance_Pct": var_captured_pct,
        "Frobenius_Residual_Error": frob_error_k,
        "Relative_Frobenius_Error_Pct": rel_frob_error_pct,
        "Next_Tail_Eigenvalue": tail_max_eig
    })

df_svd_bounds = pd.DataFrame(svd_rows)
print(df_svd_bounds.round(4).to_string(index=False))

# Non-parametric Wilcoxon signed-rank comparison of rank-(m-1), rank-m, and rank-(m+1) reconstructions
Z_hat_m_minus = (U[:, :m-1] * s[:m-1]) @ Vt[:m-1, :]
Z_hat_m = (U[:, :m] * s[:m]) @ Vt[:m, :]
Z_hat_m_plus = (U[:, :m+1] * s[:m+1]) @ Vt[:m+1, :]

obs_err_m_minus = np.linalg.norm(Z_mat - Z_hat_m_minus, axis=1)
obs_err_m = np.linalg.norm(Z_mat - Z_hat_m, axis=1)
obs_err_m_plus = np.linalg.norm(Z_mat - Z_hat_m_plus, axis=1)

w_under, p_under = stats.wilcoxon(obs_err_m_minus, obs_err_m)
w_over, p_over = stats.wilcoxon(obs_err_m, obs_err_m_plus)

df_svd_wilcoxon = pd.DataFrame([
    {
        "Comparison": f"Rank k={m-1} vs. Selected Rank k={m}",
        "Mean_Obs_Error_A": np.mean(obs_err_m_minus),
        "Mean_Obs_Error_B": np.mean(obs_err_m),
        "Wilcoxon_W": w_under,
        "Wilcoxon_p_value": p_under
    },
    {
        "Comparison": f"Selected Rank k={m} vs. Expanded Rank k={m+1}",
        "Mean_Obs_Error_A": np.mean(obs_err_m),
        "Mean_Obs_Error_B": np.mean(obs_err_m_plus),
        "Wilcoxon_W": w_over,
        "Wilcoxon_p_value": p_over
    }
])

print("\n=== WILCOXON SIGNED-RANK RECONSTRUCTION ERROR TESTS ===")
print(df_svd_wilcoxon.round(4).to_string(index=False))

# Export Step 5 Diagnostics to CSV
svd_bounds_path = os.path.join(diag_dir, "table_4_15_eckart_young_mirsky_svd_bounds.csv")
svd_wilcoxon_path = os.path.join(diag_dir, "table_4_15b_svd_rank_wilcoxon_tests.csv")
df_svd_bounds.to_csv(svd_bounds_path, index=False, encoding="utf-8")
df_svd_wilcoxon.to_csv(svd_wilcoxon_path, index=False, encoding="utf-8")
print(f"\n[SAVED] {svd_bounds_path}")
print(f"[SAVED] {svd_wilcoxon_path}")


In [ ]:
# ==============================================================================
# CELL 3: STEP 6 — MONTE CARLO ALGORITHMIC STABILITY SIMULATION (B = 10,000)
# ==============================================================================
print("=== STEP 6: MONTE CARLO ALGORITHMIC STABILITY SIMULATION (B = 10,000) ===")

# 1. Helper Functions (Aligned with Main Pipeline Architecture)
def varimax_mc(Phi, gamma=1.0, q=100, tol=1e-8):
    """Kaiser-normalized orthogonal Varimax rotation."""
    h = np.sqrt(np.sum(Phi**2, axis=1, keepdims=True))
    Phi_norm = Phi / np.maximum(h, 1e-15)
    p_rows, k_cols = Phi_norm.shape
    R_rot = np.eye(k_cols)
    d = 0.0
    for _ in range(q):
        d_old = d
        Lambda = np.dot(Phi_norm, R_rot)
        u, s_val, vh = np.linalg.svd(
            np.dot(
                Phi_norm.T,
                Lambda**3 - (gamma / p_rows) * np.dot(Lambda, np.diag(np.diag(np.dot(Lambda.T, Lambda))))
            )
        )
        R_rot = np.dot(u, vh)
        d = np.sum(s_val)
        if d_old != 0.0 and abs(d - d_old) < tol:
            break
    return np.dot(Phi_norm, R_rot) * h

def classify_tier_vec(scores):
    """Vectorized 5-tier equal-interval classifier on [0, 1] SoVI' scale."""
    tiers = np.empty(len(scores), dtype=object)
    tiers[scores <= 0.20] = "Very Low"
    tiers[(scores > 0.20) & (scores <= 0.40)] = "Low"
    tiers[(scores > 0.40) & (scores <= 0.60)] = "Moderate"
    tiers[(scores > 0.60) & (scores <= 0.80)] = "High"
    tiers[scores > 0.80] = "Very High"
    return tiers

# Resolve housing column indices dynamically for redundancy adjustment
roof_idx = next((i for i, c in enumerate(cols) if c.lower() in ["poor_roof", "poor_roof_materials"]), None)
wall_idx = next((i for i, c in enumerate(cols) if c.lower() in ["poor_wall", "poor_wall_materials"]), None)
loading_threshold = 0.35

# Baseline reference vectors loaded in Cell 1
base_sovi_norm = df_panel["SoVI_Norm"].values
base_tiers = df_panel["Tier"].values if "Tier" in df_panel.columns else classify_tier_vec(base_sovi_norm)

# 2. Monte Carlo Simulation Loop (B = 10,000 Perturbations)
B = 10_000
noise_sd = 0.05  # Controlled Gaussian perturbation (5% of 1 SD)
rng = np.random.default_rng(seed=42)

sim_scores = np.zeros((B, N))
kappa_vals = np.zeros(B)
det_vals = np.zeros(B)
rs_vals = np.zeros(B)
tier_match_vals = np.zeros(B)
mobility_vals = np.zeros(B)

for b in range(B):
    # Inject Gaussian noise and re-standardize perturbed panel
    Z_pert = Z_mat + rng.normal(loc=0.0, scale=noise_sd, size=(N, p))
    Z_pert = (Z_pert - Z_pert.mean(axis=0)) / Z_pert.std(axis=0, ddof=1)

    # Compute perturbed correlation matrix and conditioning diagnostics
    R_pert = np.corrcoef(Z_pert, rowvar=False)
    ev, evec = np.linalg.eigh(R_pert)
    idx_sort = np.argsort(ev)[::-1]
    ev, evec = ev[idx_sort], evec[:, idx_sort]

    kappa_vals[b] = ev[0] / max(ev[-1], 1e-15)
    det_vals[b] = np.linalg.det(R_pert)

    # Extract m components and apply Varimax rotation
    lu = evec[:, :m] * np.sqrt(np.maximum(ev[:m], 0.0))
    lr = varimax_mc(lu)

    # Match local rotated components to reference loadings (prevents slot-swapping)
    assigned = set()
    aligned_lr = np.zeros_like(lr)
    for g_idx in range(m):
        best_l, best_dot, best_sign = -1, -1.0, 1.0
        for l_idx in range(m):
            if l_idx in assigned:
                continue
            dot_val = np.dot(lr[:, l_idx], load_rot_ref[:, g_idx])
            if abs(dot_val) > best_dot:
                best_dot = abs(dot_val)
                best_l = l_idx
                best_sign = np.sign(dot_val) if dot_val != 0 else 1.0
        assigned.add(best_l)
        aligned_lr[:, g_idx] = lr[:, best_l] * best_sign

    # Compute thresholded component scores with housing redundancy compression
    rc_scores = np.zeros((N, m))
    for k in range(m):
        active_idx = np.where(np.abs(aligned_lr[:, k]) >= loading_threshold)[0]
        if len(active_idx) == 0:
            continue

        has_roof = roof_idx is not None and roof_idx in active_idx
        has_wall = wall_idx is not None and wall_idx in active_idx
        comp_sum = np.zeros(N)

        if has_roof and has_wall:
            s_roof = np.sign(aligned_lr[roof_idx, k])
            s_wall = np.sign(aligned_lr[wall_idx, k])
            comp_sum += (Z_pert[:, roof_idx] * s_roof + Z_pert[:, wall_idx] * s_wall) / 2.0
            for idx_v in active_idx:
                if idx_v not in (roof_idx, wall_idx):
                    comp_sum += Z_pert[:, idx_v] * np.sign(aligned_lr[idx_v, k])
            rc_scores[:, k] = comp_sum / max(len(active_idx) - 1, 1)
        else:
            for idx_v in active_idx:
                comp_sum += Z_pert[:, idx_v] * np.sign(aligned_lr[idx_v, k])
            rc_scores[:, k] = comp_sum / len(active_idx)

    # Unweighted additive aggregation and [0, 1] Min-Max rescaling
    raw_b = np.sum(rc_scores, axis=1)
    norm_b = (raw_b - raw_b.min()) / (raw_b.max() - raw_b.min() + 1e-15)
    sim_scores[b, :] = norm_b

    # Rank agreement and tier stability against baseline
    rs_vals[b] = stats.spearmanr(base_sovi_norm, norm_b).statistic
    tiers_b = classify_tier_vec(norm_b)
    match_rate = np.mean(tiers_b == base_tiers)
    tier_match_vals[b] = match_rate * 100.0
    mobility_vals[b] = 1.0 - match_rate

# 3. Compile Global Simulation Summary Table (Table 4.16)
def summarize_metric(name, arr, threshold_note):
    return {
        "Diagnostic_Metric": name,
        "Mean": np.mean(arr),
        "SD": np.std(arr, ddof=1),
        "Median": np.median(arr),
        "CI_95_Lower": np.percentile(arr, 2.5),
        "CI_95_Upper": np.percentile(arr, 97.5),
        "Stability_Benchmark": threshold_note
    }

df_mc_summary = pd.DataFrame([
    summarize_metric("Condition Number kappa(R)", kappa_vals, "kappa(R) < 30.0 (Well-conditioned)"),
    summarize_metric("Correlation Determinant |R|", det_vals, "|R| > 0 (Non-singular)"),
    summarize_metric("Spearman Rank Correlation (r_s)", rs_vals, "r_s >= 0.80 (High rank invariance)"),
    summarize_metric("Tier Classification Match (%)", tier_match_vals, ">= 85.0% exact tier agreement"),
    summarize_metric("Class Mobility Index (M_class)", mobility_vals, "M_class <= 0.15 (Low tier drift)")
])

print("\n=== TABLE 4.16: MONTE CARLO STABILITY DIAGNOSTICS SUMMARY (B = 10,000) ===")
print(df_mc_summary.round(4).to_string(index=False))

# 4. Compile Observation-Level 95% Simulation Confidence Intervals (Table 4.16b)
df_mc_obs = df_z[["Province", "Year"]].copy()
df_mc_obs["Baseline_SoVI_Norm"] = base_sovi_norm
df_mc_obs["Baseline_Tier"] = base_tiers
df_mc_obs["MC_Mean_SoVI_Norm"] = np.mean(sim_scores, axis=0)
df_mc_obs["MC_SD"] = np.std(sim_scores, axis=0, ddof=1)
df_mc_obs["MC_CI_95_Lower"] = np.percentile(sim_scores, 2.5, axis=0)
df_mc_obs["MC_CI_95_Upper"] = np.percentile(sim_scores, 97.5, axis=0)
df_mc_obs["MC_CI_Width"] = df_mc_obs["MC_CI_95_Upper"] - df_mc_obs["MC_CI_95_Lower"]

print("\n=== TABLE 4.16b: OBSERVATION-LEVEL 95% CONFIDENCE INTERVALS (FIRST 10 ROWS) ===")
print(df_mc_obs.head(10).round(4).to_string(index=False))

# 5. Export Step 6 Tables to CSV
mc_summary_path = os.path.join(diag_dir, "table_4_16_monte_carlo_stability_summary.csv")
mc_obs_path = os.path.join(diag_dir, "table_4_16b_monte_carlo_observation_ci.csv")

df_mc_summary.to_csv(mc_summary_path, index=False, encoding="utf-8")
df_mc_obs.to_csv(mc_obs_path, index=False, encoding="utf-8")

print(f"\n[DONE] Monte Carlo simulation complete (B = {B:,} iterations).")
print(f"[SAVED] {mc_summary_path}")
print(f"[SAVED] {mc_obs_path}")
